In [ ]:
from datetime import datetime, timedelta

start_date = datetime(2024, 1, 1)
end_date = datetime(2024, 12, 31)
target_weekdays = [0, 2, 4, 5]  # 월, 수, 금, 토
target_hours = ['0300', '0800', '1400', '1800']

timestamps = []

current = start_date
while current <= end_date:
    if current.weekday() in target_weekdays:
        for hour in target_hours:
            timestamps.append(current.strftime('%Y%m%d') + hour)
    current += timedelta(days=1)

print("📌 앞 5개:", timestamps[:5])
print("📌 마지막 10개:", timestamps[-10:])
print(f"✅ 총 {len(timestamps)}개의 타임스탬프가 생성되었습니다.")


📌 앞 5개: ['202401010300', '202401010800', '202401011400', '202401011800', '202401030300']
📌 마지막 10개: ['202412271400', '202412271800', '202412280300', '202412280800', '202412281400', '202412281800', '202412300300', '202412300800', '202412301400', '202412301800']
✅ 총 836개의 타임스탬프가 생성되었습니다.


In [ ]:
import requests
import pandas as pd
import xml.etree.ElementTree as ET

# 1. API 설정
SERVICE_KEY = '446e594254676b61373773624c536d'
BASE_URL = 'http://openAPI.seoul.go.kr:8088'

# 2. 호출 함수
def get_air_quality_data(date_string):
    endpoint = f'{BASE_URL}/{SERVICE_KEY}/xml/TimeAverageAirQuality/1/25/{date_string}'
    response = requests.get(endpoint)

    if response.status_code == 200:
        try:
            root = ET.fromstring(response.content)
            rows = root.findall('.//row')
            data = []
            for row in rows:
                record = {}
                for elem in row:
                    record[elem.tag] = elem.text
                record['MSRDT'] = date_string  # 수동으로 날짜 추가
                data.append(record)
            return data
        except ET.ParseError as e:
            print(f"XML 파싱 오류: {e} | 날짜: {date_string}")
            return []
    else:
        print(f"API 실패 {response.status_code} | 날짜: {date_string}")
        return []


# 반복 수집
all_data = []

for ts in timestamps:
    records = get_air_quality_data(ts)
    if records:
        all_data.extend(records)

# DataFrame으로 저장
df = pd.DataFrame(all_data)

# 6. 확인 및 저장
print(df.head())
print(f"✅ 총 {len(df)}개의 측정 기록이 수집되었습니다.")


          MSRDT MSRSTE_NM     NO2      O3    CO     SO2 PM10 PM25
0  202401010300       강남구   0.032   0.002   0.8   0.003   21   19
1  202401010300       강동구  0.0349   0.002  0.98  0.0018   20   13
2  202401010300       강북구  0.0224  0.0053  0.78   0.002   16   14
3  202401010300       강서구  0.0464  0.0025  0.95  0.0031   53   28
4  202401010300       관악구  0.0468  0.0016  1.32  0.0025   54   40
✅ 총 20800개의 측정 기록이 수집되었습니다.


In [ ]:
df.to_csv("/content/drive/MyDrive/DS teamproj/air_quality_2024_sample.csv", index=False)